# Experiment 5: Tabular Q-Learning on a Gridworld Environment
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To implement tabular Q-learning on a 5x5 Cliff Gridworld, evaluate the exploration-exploitation trade-off across exploration parameters $\epsilon \in \{0.01, 0.10, 0.20, 0.50\}$, and extract the optimal greedy policy.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

H, W = 5, 5
START = (0, 0); GOAL = (4, 4)
TRAPS = {(1, 2), (2, 2), (3, 2)} # Cliff traps

def step_gridworld(state, action):
    # Actions: 0: UP (-1,0), 1: DOWN (+1,0), 2: LEFT (0,-1), 3: RIGHT (0,+1)
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dr, dc = moves[action]
    nr = max(0, min(H - 1, state[0] + dr))
    nc = max(0, min(W - 1, state[1] + dc))
    next_s = (nr, nc)
    if next_s == GOAL: return next_s, 10.0, True
    if next_s in TRAPS: return START, -10.0, False
    return next_s, -0.1, False

def train_q_learning(eps, alpha=0.10, gamma=0.95, episodes=600, seed=42):
    rng = np.random.default_rng(seed)
    Q = np.zeros((H, W, 4))
    returns = []
    for ep in range(episodes):
        state = START; ep_ret = 0; steps = 0
        while state != GOAL and steps < 200:
            if rng.random() < eps: action = rng.integers(4)
            else: action = np.argmax(Q[state[0], state[1]])
            next_s, r, done = step_gridworld(state, action)
            best_next = np.argmax(Q[next_s[0], next_s[1]])
            td_target = r + (0.0 if done else gamma * Q[next_s[0], next_s[1], best_next])
            Q[state[0], state[1], action] += alpha * (td_target - Q[state[0], state[1], action])
            ep_ret += r; state = next_s; steps += 1
            if done: break
        returns.append(ep_ret)
    return Q, returns

eps_values = [0.01, 0.10, 0.20, 0.50]
results = {}

for eps in eps_values:
    Q, ret = train_q_learning(eps)
    results[eps] = (Q, ret)
    mean_last_100 = np.mean(ret[-100:])
    print(f"Epsilon: {eps:.2f} -> Mean Return (Last 100 Episodes): {mean_last_100:+.2f}")

# Plot Figure 5.1: Smoothed Learning Curves
fig, ax = plt.subplots(figsize=(6.5, 3.8), dpi=150)
window = 30
for eps in eps_values:
    ret = results[eps][1]
    smoothed = np.convolve(ret, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(ret)), smoothed, label=f'$\epsilon = {eps:.2f}$')

ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Return per Episode')
ax.set_title('Q-Learning Performance Across Exploration Rates $\epsilon$', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figs/fig5_1_learning_curves.png')
plt.show()

ax.set_title('Optimal Greedy Policy Map (epsilon = 0.10)', fontsize=10)
best_Q = results[0.10][0]
arrows = {0: '↑', 1: '↓', 2: '←', 3: '→'}

fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=150)
for r in range(5):
    for c in range(5):
        if (r, c) == GOAL:
            ax.add_patch(plt.Rectangle((c, 4 - r), 1, 1, facecolor='#2ca02c', alpha=0.6))
            ax.text(c + 0.5, 4 - r + 0.5, 'GOAL\n+10', ha='center', va='center', fontweight='bold', fontsize=9)
        elif (r, c) in TRAPS:
            ax.add_patch(plt.Rectangle((c, 4 - r), 1, 1, facecolor='#d62728', alpha=0.6))
            ax.text(c + 0.5, 4 - r + 0.5, 'TRAP\n-10', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
        elif (r, c) == START:
            ax.add_patch(plt.Rectangle((c, 4 - r), 1, 1, facecolor='#1f77b4', alpha=0.3))
            best_a = np.argmax(best_Q[r, c])
            ax.text(c + 0.5, 4 - r + 0.5, f"START\n{arrows[best_a]}", ha='center', va='center', fontweight='bold', fontsize=9)
        else:
            best_a = np.argmax(best_Q[r, c])
            ax.text(c + 0.5, 4 - r + 0.5, arrows[best_a], ha='center', va='center', fontsize=18)

for i in range(6):
    ax.axhline(i, color='black', linewidth=0.8)
    ax.axvline(i, color='black', linewidth=0.8)

ax.set_xlim(0, 5); ax.set_ylim(0, 5)
ax.set_xticks(np.arange(0.5, 5.5, 1)); ax.set_xticklabels(range(5))
ax.set_yticks(np.arange(0.5, 5.5, 1)); ax.set_yticklabels(range(4, -1, -1))
ax.set_xlabel('Column Index'); ax.set_ylabel('Row Index')
ax.set_title('Optimal Greedy Policy Map (epsilon = 0.10)', fontsize=10)
plt.tight_layout()
plt.savefig('figs/fig5_2_greedy_policy.png')
plt.show()
